# Complete Audio Pipeline Test

This notebook demonstrates the full audio capture and processing pipeline.

## What this implements
1. **Continuous listening** with WebRTC VAD
2. **Speech detection** using buffered frames
3. **Silence trimming** to remove dead air
4. **Audio normalization** to standardize volume
5. **Clean audio output** ready for STT

## This became the production pipeline in `Yumi_Hears/pipeline.py`

## Key parameters
- Sample rate: 16kHz (required for VAD)
- Frame duration: 30ms
- VAD aggressiveness: Mode 2 (balanced)
- Speech trigger: 6/15 frames
- Silence trigger: 12/15 frames

## Dependencies
```bash
pip install sounddevice webrtcvad-wheels numpy scipy
```

In [ ]:
import sounddevice as sd
import numpy as np
import webrtcvad
import collections
import wave
from scipy.signal import resample

# Audio configuration
RATE = 16000              # 16kHz required by WebRTC VAD
FRAME_DURATION = 30       # 30ms frames (20, 30, or 60ms only)
FRAME_SIZE = int(RATE * FRAME_DURATION / 1000)  # 480 samples
CHANNELS = 1              # Mono audio

print(f"Audio config: {RATE}Hz, {FRAME_DURATION}ms frames, {FRAME_SIZE} samples/frame")

In [ ]:
RATE = 16000
FRAME_DURATION = 30  # ms
FRAME_SIZE = int(RATE * FRAME_DURATION / 1000)
CHANNELS = 1
FORMAT = 'int16'

In [ ]:
def float_to_pcm16(audio):

    audio = np.clip(audio, -1, 1)

    audio_int16 = (audio * 32767).astype(np.int16)

    return audio_int16

In [ ]:
def save_wav(filename, audio):

    with wave.open(filename, 'wb') as wf:

        wf.setnchannels(1)
        wf.setsampwidth(2)
        wf.setframerate(RATE)

        wf.writeframes(audio.tobytes())

In [ ]:
def remove_silence(audio):

    vad = webrtcvad.Vad(2)

    frames = []

    num_frames = len(audio) // FRAME_SIZE

    for i in range(num_frames):

        frame = audio[i*FRAME_SIZE:(i+1)*FRAME_SIZE]

        if len(frame) < FRAME_SIZE:
            continue

        is_speech = vad.is_speech(frame.tobytes(), RATE)

        if is_speech:
            frames.append(frame)

    if len(frames) == 0:
        return audio

    return np.concatenate(frames)

In [ ]:
def normalize_audio(audio):

    max_val = np.max(np.abs(audio))

    if max_val == 0:
        return audio

    audio = audio / max_val

    audio = audio * 0.9

    return audio.astype(np.int16)

In [ ]:
def listen_for_speech():
    """
    Continuously listen for speech using WebRTC VAD.
    
    Uses a ring buffer to detect speech start/end:
    - Speech starts: 6/15 frames are speech
    - Speech ends: 12/15 frames are silence
    
    Returns:
        numpy array of recorded audio (int16)
    """
    vad = webrtcvad.Vad(2)  # Aggressiveness mode 0-3
    print("Listening...")
    
    # Ring buffer for VAD decisions
    buffer = collections.deque(maxlen=10)
    recording = []
    triggered = False
    
    while True:
        # Capture audio frame
        audio = sd.rec(FRAME_SIZE, samplerate=RATE, channels=1)
        sd.wait()
        audio = audio.flatten()
        pcm = float_to_pcm16(audio)
        
        # Check if frame contains speech
        is_speech = vad.is_speech(pcm.tobytes(), RATE)
        
        if not triggered:
            # Not yet triggered - check buffer for speech start
            buffer.append((pcm, is_speech))
            if sum([f[1] for f in buffer]) > 7:
                triggered = True
                print("Speech started")
                recording.extend([f[0] for f in buffer])
                buffer.clear()
        else:
            # Triggered - record and check for speech end
            recording.append(pcm)
            buffer.append((pcm, is_speech))
            if sum([not f[1] for f in buffer]) > 7:
                print("Speech ended")
                break
    
    return np.concatenate(recording)

In [ ]:
def capture_and_clean():

    audio = listen_for_speech()

    print("Raw length:", len(audio))

    audio = remove_silence(audio)

    print("After silence removal:", len(audio))

    audio = normalize_audio(audio)

    save_wav("clean_speech.wav", audio)

    print("Saved clean_speech.wav")

    return audio

In [ ]:
audio = capture_and_clean()